# SITS - Classificacao de Imagens

Este notebook aplica modelos treinados para classificar imagens de serie temporal.

**Entrada:** 
- Stack temporal (GeoTIFF com bandas = timesteps x channels)
- Modelo treinado (do notebook 04_treinamento)

**Saida:**
- Mapa classificado (GeoTIFF)

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Imports do modulo
from sits.classification import (
    ExperimentManager,
    classify_from_experiment,
)

---
## 2. Configuracao

In [ ]:
# =============================================================================
# CONFIGURACAO - EDITE AQUI
# =============================================================================

# Caminho para a sessao/experimento
SESSION_PATH = "../sessions/seu_projeto"
EXPERIMENT_NAME = "exp_v1"

# -----------------------------------------------------------------------------
# Modelo a usar
# -----------------------------------------------------------------------------
# Opcoes:
#   - "auto" para usar o melhor modelo automaticamente
#   - Nome especifico como "inception_time__default"
MODEL_TO_USE = "auto"

# -----------------------------------------------------------------------------
# Imagem de entrada
# -----------------------------------------------------------------------------
# Caminho para o stack temporal (GeoTIFF ou ENVI)
# Bandas devem estar no formato: B1T1, B2T1, B3T1, B4T1, B1T2, B2T2, ...
# Onde B = banda e T = timestep
IMAGE_PATH = "../data/stack_temporal.tif"

# -----------------------------------------------------------------------------
# Imagem de saida
# -----------------------------------------------------------------------------
OUTPUT_FILENAME = "classificacao.tif"

# -----------------------------------------------------------------------------
# Parametros do stack
# -----------------------------------------------------------------------------
NUM_TIMESTEPS = 12  # Numero de datas na serie temporal

# -----------------------------------------------------------------------------
# Parametros de processamento
# -----------------------------------------------------------------------------
CHUNK_SIZE = 2000   # Tamanho do chunk em pixels (ajuste conforme memoria GPU)
BATCH_SIZE = 2000   # Tamanho do batch para inferencia

## 3. Carregar Experimento e Selecionar Modelo

In [ ]:
# Carregar experimento
experiment_dir = Path(SESSION_PATH) / "training" / EXPERIMENT_NAME

if not experiment_dir.exists():
    raise FileNotFoundError(f"Experimento nao encontrado: {experiment_dir}")

exp = ExperimentManager.load(experiment_dir)
print(f"Experimento carregado: {experiment_dir}")

# Obter resumo dos modelos
summary = exp.summary()
print(f"Modelos treinados: {len(summary)}")

In [ ]:
# Selecionar modelo
if MODEL_TO_USE == "auto":
    model_name, model_path = exp.get_best_model()
    print(f"Melhor modelo selecionado: {model_name}")
else:
    model_name = MODEL_TO_USE
    model_path = exp.models_dir / model_name
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo nao encontrado: {model_path}")
    print(f"Modelo selecionado: {model_name}")

# Mostrar metricas do modelo selecionado
model_metrics = summary[summary["model"] == model_name].iloc[0]
print(f"\nMetricas do modelo:")
print(f"  Accuracy: {model_metrics['accuracy']:.4f}")
print(f"  F1-Score: {model_metrics['f1_score']:.4f}")

## 4. Verificar Imagem de Entrada

In [ ]:
# Verificar se o arquivo existe
image_path = Path(IMAGE_PATH)

if not image_path.exists():
    raise FileNotFoundError(f"Imagem nao encontrada: {image_path}")

# Ler metadados
with rasterio.open(image_path) as src:
    print(f"Imagem: {image_path}")
    print(f"  Dimensoes: {src.width} x {src.height}")
    print(f"  Bandas: {src.count}")
    print(f"  CRS: {src.crs}")
    print(f"  Dtype: {src.dtypes[0]}")
    
    num_channels_img = src.count // NUM_TIMESTEPS
    print(f"\n  {src.count} bandas = {NUM_TIMESTEPS} timesteps x {num_channels_img} canais")

## 5. Carregar Informacoes das Classes

In [ ]:
# Carregar mapeamento de classes do experimento
class_mapping_file = exp.data_dir / "class_mapping.json"

index_to_class = None

if class_mapping_file.exists():
    with open(class_mapping_file, "r", encoding="utf-8") as f:
        class_data = json.load(f)
    
    # Suporta diferentes estruturas de arquivo
    if "idx_to_name" in class_data:
        index_to_class = {int(k): v for k, v in class_data["idx_to_name"].items()}
    elif "class_mapping" in class_data:
        index_to_class = {v: k for k, v in class_data["class_mapping"].items()}
    else:
        index_to_class = {v: k for k, v in class_data.items()}
    
    print("Mapeamento de classes:")
    print("-" * 30)
    for idx in sorted(index_to_class.keys()):
        print(f"  {idx}: {index_to_class[idx]}")
else:
    print("AVISO: Arquivo de mapeamento de classes nao encontrado.")

## 6. Executar Classificacao

In [ ]:
# Construir caminho de saida
inference_dir = Path(SESSION_PATH) / "training" / EXPERIMENT_NAME / "inference"
inference_dir.mkdir(parents=True, exist_ok=True)

# Pasta para figuras
figures_dir = inference_dir / "figures"
figures_dir.mkdir(exist_ok=True)

output_path = inference_dir / OUTPUT_FILENAME

print(f"Configuracao da classificacao:")
print(f"  Modelo: {model_name}")
print(f"  Entrada: {image_path}")
print(f"  Saida: {output_path}")
print(f"  Chunk size: {CHUNK_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")

In [ ]:
# Executar classificacao
print("\nIniciando classificacao...")

classify_from_experiment(
    experiment_dir=experiment_dir,
    model_name=model_name,
    image_path=image_path,
    output_path=output_path,
    num_timesteps=NUM_TIMESTEPS,
    chunk_size=CHUNK_SIZE,
    batch_size=BATCH_SIZE,
    verbose=True,
)

print(f"\nClassificacao concluida!")
print(f"Arquivo salvo em: {output_path}")

## 7. Visualizar Resultado

In [ ]:
# Carregar resultado
with rasterio.open(output_path) as src:
    classified = src.read(1)
    profile = src.profile

# Estatisticas
unique, counts = np.unique(classified, return_counts=True)
total_pixels = classified.size

print("Estatisticas da classificacao:")
print("-" * 55)
for idx, count in zip(unique, counts):
    if index_to_class:
        class_name = index_to_class.get(idx, f"Classe {idx}")
    else:
        class_name = f"Classe {idx}"
    pct = count / total_pixels * 100
    print(f"  {idx}: {class_name:25s} - {count:10d} px ({pct:5.2f}%)")
print("-" * 55)
print(f"  Total: {total_pixels:10d} pixels")

In [ ]:
# Visualizacao do mapa
fig, ax = plt.subplots(figsize=(12, 12))

# Criar colormap
n_classes = len(unique)
if n_classes <= 10:
    cmap = plt.cm.get_cmap('tab10', n_classes)
else:
    cmap = plt.cm.get_cmap('tab20', n_classes)

# Normalizacao
norm = plt.Normalize(vmin=unique.min() - 0.5, vmax=unique.max() + 0.5)

# Plot
im = ax.imshow(classified, cmap=cmap, norm=norm)

# Colorbar
cbar = plt.colorbar(im, ax=ax, ticks=unique, shrink=0.7)
if index_to_class:
    cbar.ax.set_yticklabels([index_to_class.get(i, f"{i}") for i in unique])
cbar.ax.invert_yaxis()

ax.set_title(f"Classificacao - Modelo: {model_name}")
ax.set_xlabel("Coluna (pixel)")
ax.set_ylabel("Linha (pixel)")

plt.tight_layout()

# Salvar figura
fig_path = figures_dir / (output_path.stem + ".png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Figura salva em: {fig_path}")

plt.show()

In [ ]:
# Grafico de distribuicao
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Labels
if index_to_class:
    labels = [index_to_class.get(i, f"{i}") for i in unique]
else:
    labels = [f"Classe {i}" for i in unique]

# Cores
colors = [cmap(norm(i)) for i in unique]

# Grafico de barras
bars = ax1.bar(labels, counts, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xlabel("Classe")
ax1.set_ylabel("Numero de Pixels")
ax1.set_title("Distribuicao das Classes")
ax1.tick_params(axis='x', rotation=45)

# Adicionar valores
for bar, count in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{count:,}", ha='center', va='bottom', fontsize=8, rotation=90)

# Grafico de pizza
ax2.pie(counts, labels=labels, colors=colors, autopct='%1.1f%%',
        startangle=90, explode=[0.02] * len(unique))
ax2.set_title("Proporcao por Classe")

plt.tight_layout()
plt.show()

---
## 8. Classificacao em Lote (Opcional)

Para classificar multiplas imagens de uma vez.

In [ ]:
# Lista de imagens para classificar em lote
# Descomente e edite conforme necessario

# IMAGES_TO_CLASSIFY = [
#     {
#         "input": "../data/stack_area1.tif",
#         "output": "classificado_area1.tif",
#     },
#     {
#         "input": "../data/stack_area2.tif",
#         "output": "classificado_area2.tif",
#     },
# ]
# 
# for i, img_info in enumerate(IMAGES_TO_CLASSIFY):
#     print(f"\n[{i+1}/{len(IMAGES_TO_CLASSIFY)}] Classificando: {img_info['input']}")
#     
#     out_path = inference_dir / img_info["output"]
#     
#     classify_from_experiment(
#         experiment_dir=experiment_dir,
#         model_name=model_name,
#         image_path=img_info["input"],
#         output_path=out_path,
#         num_timesteps=NUM_TIMESTEPS,
#         chunk_size=CHUNK_SIZE,
#         batch_size=BATCH_SIZE,
#         verbose=True,
#     )
# 
# print("\nTodas as imagens foram classificadas!")

---
## 9. Comparar Modelos (Opcional)

Classificar com diferentes modelos e comparar resultados.

In [ ]:
# Comparar top 3 modelos
# Descomente para executar

# top_models = summary.head(3)["model"].tolist()
# 
# for model in top_models:
#     out_path = inference_dir / f"classificado_{model}.tif"
#     print(f"\nClassificando com {model}...")
#     
#     classify_from_experiment(
#         experiment_dir=experiment_dir,
#         model_name=model,
#         image_path=image_path,
#         output_path=out_path,
#         num_timesteps=NUM_TIMESTEPS,
#         chunk_size=CHUNK_SIZE,
#         batch_size=BATCH_SIZE,
#         verbose=True,
#     )
#     print(f"Salvo em: {out_path}")

---

## Conclusao

A classificacao foi concluida!

**Arquivos gerados:**
- Mapa classificado (GeoTIFF) na pasta `inference/`
- Visualizacao (PNG) na pasta `inference/figures/`

Os valores dos pixels correspondem aos indices das classes no mapeamento.